In [1]:
#Plant disease detector
#This model is bulit by me for my Village farmers  who lose crops to diseases which they dont identify inculding my father

#Farmers sends a photo in whatsapp, This tool predicts the disease and sends the disease name and treatment for the respective disease
#It higlights the affected area through a heatmap so the farmers identify where the model is looking
#this notebook is only on this model it does not include whatsapp integration

In [16]:
!pip install -U kaggle split-folders gradio -q
#for file handling and data
import os
import pandas as pd
import numpy as np
import shutil

#for training  deep learning model
import tensorflow as tf
print("tensorflow imported sucessfully")

#for image processing and visualization
import cv2
import matplotlib.pyplot as plt
from PIL import Image

#for dataset splitting and app deployement
import splitfolders
print("splitfolders imported sucessfully")# just conforming colab has a litte problem with splitfolders
import gradio as gr

from google.colab import drive

#keras Stuff for building model
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet_v2 import EfficientNetV2B0

#just checking versions so i know nothing broke
print("tensorflow:" , tf.__version__)
print("gradio", gr.__version__)



tensorflow imported sucessfully
splitfolders imported sucessfully
tensorflow: 2.20.0
gradio 6.20.0


In [30]:
drive.mount("/content/drive")

# making all my folders in my drive so i dont lose when my model when colab disconnects
project_dir="/content/drive/MyDrive/PlantDiseaseProject"

if not os.path.exists(project_dir):
    os.makedirs(project_dir)

#folders inside for models,outputs and samples
model_dir="/content/drive/MyDrive/PlantDiseaseProject/models"
output_dir = "/content/drive/MyDrive/PlantDiseaseProject/outputs"
sample_dir = "/content/drive/MyDrive/PlantDiseaseProject/samples"

os.makedirs(model_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)
os.makedirs(sample_dir, exist_ok=True)

# paths to save the model. FINAL one is after all epochs, BEST one is the best score during training
final_model_path = model_dir + "/efficientnetv2_plant_disease_model.keras"
best_model_path = model_dir + "/best_efficientnetv2_plant_disease_model.keras"

print("folders done")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
folders done


In [39]:
#setting up kaggle so i can download the dataset
#i need the kaggle.json api key file for this  i put mine in drive

kaggle_json_drive = "/content/drive/MyDrive/Kaggle/kaggle.json"
kaggle_json_local = "/root/.kaggle/kaggle.json"

os.makedirs("/root/.kaggle", exist_ok=True)

#copied the file from drive to the localkaggle folder so kaggle can find it
if os.path.exists(kaggle_json_drive):
  shutil.copy(kaggle_json_drive, kaggle_json_local)
  os.chmod(kaggle_json_local,0o600)# kaggle throws a warning if the file isnt 600 permissions, took me a while to figure that out

  print("kaggle.json loaded")
else:
  print("not found")


kaggle.json loaded


In [42]:
# downloading plant village dataset form kaggle
#only downloads when i need it(saves time when i rerun notebook)

raw_data_dir = "/content/plantvillage"
if not os.path.exists(raw_data_dir):
  #unzipping so it extracts automatically or i get a zip and have to unzip it myself
  !kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/plantvillage --unzip
  print("dataset downloaded")
else:
  print("data set alrady there skipping downlaoding")

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:11<00:00, 190MB/s]

dataset downloaded


In [48]:
#splitting the dataset into three folders train,val,test to prevent data leakage
# i am using colour images(there is also balck and white images and and segmented versions)

source_dir = "/content/plantvillage/plantvillage dataset/color"
split_data_dir = "/content/plantvillage_split"

if not os.path.exists(split_data_dir):
  splitfolders.ratio(
      source_dir,
      output=split_data_dir,
      seed=42,# seed 42 because i get exact result if i rerun anytime
      ratio=(0.70,0.15,0.15),# best to mantain balance
      group_prefix=None,
      move=False

  )
  print("data splitted")
else:
  print("data already splitted")

Copying files: 54305 files [00:11, 4881.75 files/s]

data splitted
